# VisionBridge A-Z base model training (no-fail Colab)

Run cells **top → bottom** on a **GPU** runtime if possible.

This notebook:
1. Clones VisionBridge and builds an isolated **Python 3.12** env with `uv`
2. Installs **NumPy 2.1.3**, **MediaPipe 0.10.35**, **PyTorch 2.9.0** (CUDA then CPU fallback)
3. Downloads + SHA-256 verifies the MediaPipe Hand Landmarker model
4. Downloads + SHA-256 verifies RealSign `Dataset.zip` from the pinned commit
5. Extracts 126D landmarks (stratified train/val; test left untouched)
6. Checks the V3 architecture contract (126 → 128 → 64 → 26)
7. Trains up to 1000 epochs and evaluates the test split

**Fixes baked in:** `MPLBACKEND=Agg` (Colab matplotlib_inline leak), pip bootstrap in venv, torch install without clobbering NumPy, full subprocess stdout/stderr on every failure.

In [ ]:
# Cell 1 — Clean workspace + Python 3.12 env + deps
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time

ROOT = Path("/content")
os.chdir(ROOT)

REPO = ROOT / "VisionBridge"
VENV = ROOT / "visionbridge_train_env"
PYTHON = VENV / "bin" / "python"

# Prevent Colab's matplotlib_inline backend from breaking MediaPipe imports
# inside the isolated venv (matplotlib_inline is not installed there).
os.environ["MPLBACKEND"] = "Agg"


def run(cmd, env=None, check=True, cwd=None):
    cmd = [str(value) for value in cmd]
    print("$", " ".join(cmd), flush=True)
    result = subprocess.run(
        cmd,
        check=False,
        env=env,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"  {' '.join(cmd)}\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )
    return result


def run_retry(cmd, attempts=3, env=None, cwd=None):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            print(f"Attempt {attempt}/{attempts}", flush=True)
            return run(cmd, env=env, cwd=cwd)
        except Exception as exc:
            last_error = exc
            print("Attempt failed:", repr(exc), flush=True)
            if attempt < attempts:
                time.sleep(5 * attempt)
    raise RuntimeError(
        f"Command failed after {attempts} attempts: " + " ".join(str(x) for x in cmd)
    ) from last_error


def make_venv_env():
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO / "backend")
    env["MPLBACKEND"] = "Agg"
    env["PYTHONUNBUFFERED"] = "1"
    # Avoid leaking Colab notebook backend into child matplotlib
    env.pop("MPLCONFIGDIR", None)
    return env


if REPO.exists():
    print("Removing previous /content/VisionBridge workspace...")
    shutil.rmtree(REPO)

if VENV.exists():
    print("Removing previous training environment...")
    shutil.rmtree(VENV)

print("\nCloning VisionBridge...")
run_retry(
    [
        "git", "clone", "--depth", "1",
        "https://github.com/BharathWaj-K-R/VisionBridge.git",
        str(REPO),
    ],
    attempts=3,
)
if not (REPO / ".git").is_dir():
    raise RuntimeError("VisionBridge clone did not produce a valid Git repository.")
print("VisionBridge source workspace: PASS")

print("\nInstalling uv...")
run_retry(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "--retries", "5", "--timeout", "120", "uv",
    ],
    attempts=3,
)
UV = [sys.executable, "-m", "uv"]

print("\nEnsuring Python 3.12 is available...")
run_retry([*UV, "python", "install", "3.12"], attempts=3)

print("\nCreating isolated training environment...")
run([*UV, "venv", "--python", "3.12", str(VENV)])
if not PYTHON.is_file():
    raise RuntimeError(f"Training Python was not created: {PYTHON}")
print("Training Python:", PYTHON)

# Bootstrap pip inside the venv (needed for torch index installs)
print("\nBootstrapping pip in the training venv...")
run_retry([*UV, "pip", "install", "--python", str(PYTHON), "pip", "setuptools", "wheel"], attempts=3)

gpu_probe = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
has_gpu = gpu_probe.returncode == 0
print("GPU available:", has_gpu)

print("\nInstalling NumPy 2.1.3 + MediaPipe 0.10.35...")
run_retry(
    [
        *UV, "pip", "install", "--python", str(PYTHON), "--reinstall",
        "numpy==2.1.3", "mediapipe==0.10.35",
    ],
    attempts=3,
)

# PyTorch: try CUDA indexes first, then CPU. Use --extra-index-url so PyPI
# deps stay available, and re-pin numpy afterward so torch cannot clobber it.
PIP = [str(PYTHON), "-m", "pip"]
torch_candidates = []
if has_gpu:
    torch_candidates.extend([
        ("CUDA 12.8", "https://download.pytorch.org/whl/cu128"),
        ("CUDA 12.6", "https://download.pytorch.org/whl/cu126"),
    ])
torch_candidates.append(("CPU", "https://download.pytorch.org/whl/cpu"))

torch_installed = False
torch_errors = []
child_env = make_venv_env()

for label, index_url in torch_candidates:
    print(f"\nAttempting PyTorch 2.9.0 ({label})...")
    result = subprocess.run(
        [
            *PIP, "install", "--disable-pip-version-check", "--no-cache-dir",
            "--retries", "5", "--timeout", "120",
            "torch==2.9.0",
            "--extra-index-url", index_url,
        ],
        check=False,
        text=True,
        capture_output=True,
        env=child_env,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)

    if result.returncode == 0:
        # Re-pin numpy in case a dependency shifted it
        subprocess.run(
            [*PIP, "install", "--disable-pip-version-check", "--quiet", "numpy==2.1.3"],
            check=False, text=True, capture_output=True, env=child_env,
        )
        import_test = subprocess.run(
            [
                str(PYTHON), "-c",
                (
                    "import os; os.environ['MPLBACKEND']='Agg'; "
                    "import torch; import numpy as np; "
                    "print('torch_version=' + torch.__version__); "
                    "print('numpy_version=' + np.__version__); "
                    "print('cuda_available=' + str(torch.cuda.is_available())); "
                    "assert torch.__version__.split('+')[0] == '2.9.0'; "
                    "assert np.__version__ == '2.1.3'"
                ),
            ],
            check=False,
            text=True,
            capture_output=True,
            env=child_env,
        )
        if import_test.stdout:
            print(import_test.stdout, end="")
        if import_test.stderr:
            print(import_test.stderr, end="", file=sys.stderr)
        if import_test.returncode == 0:
            torch_installed = True
            print(f"PyTorch 2.9.0 ({label}): PASS")
            break

    error_text = result.stderr or result.stdout or "unknown installation error"
    torch_errors.append(f"{label}: {error_text}")
    print(f"PyTorch {label} installation did not succeed; trying the next supported build.")

if not torch_installed:
    raise RuntimeError(
        "No supported PyTorch 2.9.0 build could be installed.\n\n" + "\n\n".join(torch_errors)
    )

ENV = make_venv_env()
version_check = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
import sys
import torch
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

assert sys.version_info[:2] == (3, 12), sys.version_info
assert np.__version__ == "2.1.3", np.__version__
assert mp.__version__ == "0.10.35", mp.__version__
print("MediaPipe Tasks API:", vision.HandLandmarker.__name__)
print("Training environment: PASS")
'''
run([str(PYTHON), "-c", version_check], env=ENV)
print("\nCell 1 complete.")

In [ ]:
# Cell 2 — Download and verify MediaPipe Hand Landmarker .task model
from pathlib import Path
import hashlib
import time
import urllib.request

HAND_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
HAND_MODEL_PATH = Path("/content/hand_landmarker.task")
HAND_MODEL_SIZE = 7_819_105
HAND_MODEL_SHA256 = "fbc2a30080c3c557093b5ddfc334698132eb341044ccee322ccf8bcf3607cde1"
MAX_ATTEMPTS = 4

if HAND_MODEL_PATH.exists():
    HAND_MODEL_PATH.unlink()
PARTIAL = HAND_MODEL_PATH.with_suffix(".task.part")
if PARTIAL.exists():
    PARTIAL.unlink()

last_error = None
for attempt in range(1, MAX_ATTEMPTS + 1):
    try:
        print(f"Downloading MediaPipe Hand Landmarker (attempt {attempt}/{MAX_ATTEMPTS})...")
        request = urllib.request.Request(
            HAND_MODEL_URL,
            headers={"User-Agent": "VisionBridge-V3-training/1.0"},
        )
        with urllib.request.urlopen(request, timeout=180) as response, PARTIAL.open("wb") as output:
            while True:
                chunk = response.read(8 * 1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
        PARTIAL.replace(HAND_MODEL_PATH)
        last_error = None
        break
    except Exception as exc:
        last_error = exc
        print("Download attempt failed:", repr(exc))
        if PARTIAL.exists():
            PARTIAL.unlink()
        if attempt < MAX_ATTEMPTS:
            time.sleep(3 * attempt)

if last_error is not None:
    raise RuntimeError(
        f"MediaPipe Hand Landmarker download failed after {MAX_ATTEMPTS} attempts."
    ) from last_error

size = HAND_MODEL_PATH.stat().st_size
digest = hashlib.sha256()
with HAND_MODEL_PATH.open("rb") as file:
    while True:
        chunk = file.read(8 * 1024 * 1024)
        if not chunk:
            break
        digest.update(chunk)
sha256 = digest.hexdigest()

if size != HAND_MODEL_SIZE:
    raise RuntimeError(f"Hand model size mismatch: expected {HAND_MODEL_SIZE}, got {size}")
if sha256 != HAND_MODEL_SHA256:
    raise RuntimeError(
        f"Hand model SHA-256 mismatch: expected {HAND_MODEL_SHA256}, got {sha256}"
    )

print("Hand model bytes:", size)
print("Hand model SHA-256:", sha256)
print("Hand model integrity: PASS")

In [ ]:
# Cell 3 — Verify Hand Landmarker loads in the training env
import os
import subprocess
import sys
from pathlib import Path

PYTHON = Path("/content/visionbridge_train_env/bin/python")
MODEL = Path("/content/hand_landmarker.task")

if not PYTHON.exists():
    raise RuntimeError("Training environment is missing. Run Cell 1.")
if not MODEL.exists():
    raise RuntimeError("MediaPipe hand model is missing. Run Cell 2.")

ENV = os.environ.copy()
ENV["MPLBACKEND"] = "Agg"

smoke = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
import sys
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = sys.argv[1]
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
)
detector = vision.HandLandmarker.create_from_options(options)
detector.close()
print("Hand Landmarker initialization: PASS")
'''

result = subprocess.run(
    [str(PYTHON), "-c", smoke, str(MODEL)],
    check=False, env=ENV, text=True, capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        "Hand Landmarker smoke test failed.\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
print("Cell 3 complete.")

In [ ]:
# Cell 4 — Download, verify, extract RealSign Dataset.zip (pinned commit)
from pathlib import Path
import hashlib
import shutil
import time
import urllib.request
import zipfile

DATASET_DIR = Path("/content/RealSign")
ZIP_PATH = Path("/content/RealSign_Dataset.zip")
SOURCE_REPOSITORY = "https://github.com/RealSign62/RealSign-Indian-Sign-Language-Dataset"
SOURCE_COMMIT = "17c51dcc158b7b6359b7c6667edc51c7f148f3f4"
SOURCE_URL = (
    "https://media.githubusercontent.com/media/"
    "RealSign62/RealSign-Indian-Sign-Language-Dataset/"
    f"{SOURCE_COMMIT}/Dataset.zip"
)
EXPECTED_SIZE = 656_689_688
EXPECTED_SHA256 = "008cae248e346b8c31fbbea057fcc3f69c6909d29a88e6bb1fb0369f528de2b5"
MAX_ATTEMPTS = 5

if DATASET_DIR.exists():
    print("Removing previous RealSign extraction...")
    shutil.rmtree(DATASET_DIR)
if ZIP_PATH.exists():
    print("Removing previous RealSign archive...")
    ZIP_PATH.unlink()
PARTIAL_PATH = ZIP_PATH.with_suffix(".zip.part")
if PARTIAL_PATH.exists():
    PARTIAL_PATH.unlink()

print("RealSign source:", SOURCE_REPOSITORY)
print("Direct download URL:", SOURCE_URL)

last_error = None
for attempt in range(1, MAX_ATTEMPTS + 1):
    try:
        print(f"\nDownloading RealSign Dataset.zip (attempt {attempt}/{MAX_ATTEMPTS})...")
        request = urllib.request.Request(
            SOURCE_URL,
            headers={"User-Agent": "VisionBridge-V3-training/1.0"},
        )
        with urllib.request.urlopen(request, timeout=300) as response, PARTIAL_PATH.open("wb") as output:
            downloaded = 0
            while True:
                chunk = response.read(8 * 1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                downloaded += len(chunk)
                if downloaded >= 64 * 1024 * 1024 and downloaded % (64 * 1024 * 1024) < 8 * 1024 * 1024:
                    print(f"Downloaded {downloaded:,} bytes", flush=True)
        PARTIAL_PATH.replace(ZIP_PATH)
        last_error = None
        break
    except Exception as exc:
        last_error = exc
        print("RealSign download attempt failed:", repr(exc))
        if PARTIAL_PATH.exists():
            PARTIAL_PATH.unlink()
        if attempt < MAX_ATTEMPTS:
            time.sleep(5 * attempt)

if last_error is not None:
    raise RuntimeError(
        f"RealSign Dataset.zip download failed after {MAX_ATTEMPTS} attempts."
    ) from last_error

size = ZIP_PATH.stat().st_size
if size != EXPECTED_SIZE:
    raise RuntimeError(f"RealSign archive size mismatch: expected {EXPECTED_SIZE}, got {size}")

sha256 = hashlib.sha256()
with ZIP_PATH.open("rb") as file:
    while True:
        chunk = file.read(8 * 1024 * 1024)
        if not chunk:
            break
        sha256.update(chunk)
digest = sha256.hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(
        f"RealSign archive SHA-256 mismatch: expected {EXPECTED_SHA256}, got {digest}"
    )

if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError("Downloaded RealSign file is not a valid ZIP archive.")

with zipfile.ZipFile(ZIP_PATH) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity check failed at member: {bad_member}")
    archive.extractall(DATASET_DIR)

expected = {"Training": False, "Validation": False, "Testing": False}
for path in DATASET_DIR.rglob("*"):
    if not path.is_dir():
        continue
    normalized = path.name.strip().lower()
    if normalized in {"training", "training (a-z)"}:
        expected["Training"] = True
    elif normalized in {"validation", "validation (a-z)", "val"}:
        expected["Validation"] = True
    elif normalized in {"testing", "testing (a-z)", "test"}:
        expected["Testing"] = True

missing = [name for name, exists in expected.items() if not exists]
if missing:
    raise RuntimeError(
        "RealSign extraction is missing expected split folders: " + ", ".join(missing)
    )

print("\nRealSign archive bytes:", size)
print("RealSign archive SHA-256:", digest)
print("Training / Validation / Testing folders: PASS")
print("RealSign direct download + extraction: PASS")

In [ ]:
# Cell 5 — Extract 126D landmarks (slow: 10–40+ min on CPU)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"
ENV["PYTHONUNBUFFERED"] = "1"

out_dir = Path("/content/visionbridge_letter_data")
if not PYTHON.exists():
    raise RuntimeError("Training Python is missing. Run Cell 1.")
if not Path("/content/RealSign").is_dir():
    raise RuntimeError("RealSign dataset is missing. Run Cell 4.")
if not Path("/content/hand_landmarker.task").is_file():
    raise RuntimeError("Hand model is missing. Run Cell 2.")

if out_dir.exists():
    import shutil
    shutil.rmtree(out_dir)

command = [
    str(PYTHON),
    str(REPO / "backend/scripts/prepare_letter_dataset.py"),
    "--input-root", "/content/RealSign",
    "--output-dir", str(out_dir),
    "--validation-ratio", "0.20",
    "--seed", "42",
    "--hand-model-path", "/content/hand_landmarker.task",
]

print("Running landmark preparation (this is the slow step)...", flush=True)
result = subprocess.run(command, check=False, env=ENV, text=True, capture_output=True)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)

required = [
    out_dir / "train.npz",
    out_dir / "val.npz",
    out_dir / "test.npz",
    out_dir / "labels.json",
    out_dir / "duplicate_report.json",
]
if result.returncode != 0:
    raise RuntimeError(
        f"prepare_letter_dataset.py failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
missing_outputs = [str(path) for path in required if not path.is_file()]
if missing_outputs:
    raise RuntimeError(
        "Landmark preparation finished but required outputs are missing:\n"
        + "\n".join(missing_outputs)
    )
print("Landmark preparation: PASS")

In [ ]:
# Cell 6 — Validate prepared NPZ + duplicate-leakage contract
from pathlib import Path
import json
import numpy as np

root = Path("/content/visionbridge_letter_data")
metadata = json.loads((root / "labels.json").read_text(encoding="utf-8"))

labels = metadata["labels"]
if labels != list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    raise RuntimeError(f"Expected A-Z labels, got {labels}")

print("Labels:", "".join(labels))
print("Split policy:", metadata["split_policy"])

for split in ("train", "val", "test"):
    data = np.load(root / f"{split}.npz")
    x = data["x"]
    y = data["y"]
    if x.ndim != 2 or x.shape[1] != 126:
        raise RuntimeError(f"{split} has invalid shape: {x.shape}")
    if y.ndim != 1 or len(x) != len(y) or len(x) == 0:
        raise RuntimeError(f"{split} has invalid labels")
    if not np.isfinite(x).all():
        raise RuntimeError(f"{split} contains NaN or Inf")
    counts = [int((y == i).sum()) for i in range(26)]
    if any(count == 0 for count in counts):
        raise RuntimeError(f"{split} is missing an A-Z class: {counts}")
    print(f"{split}: samples={len(x)} shape={x.shape}")

duplicate_report = json.loads((root / "duplicate_report.json").read_text(encoding="utf-8"))
if duplicate_report["train_val_overlap_hashes"]:
    raise RuntimeError("Exact duplicate image hashes leaked between train and validation")
if duplicate_report["train_test_overlap_hashes"] or duplicate_report["val_test_overlap_hashes"]:
    raise RuntimeError("Exact duplicate image hashes still overlap the untouched source test split")

print("Duplicate-leakage contract: PASS")
print("Prepared dataset contract: PASS")

In [ ]:
# Cell 7 — Verify VisionBridge V3 architecture contract
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

contract = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
import torch
from app.models.letter_model import (
    INPUT_DIM, HIDDEN_DIM, EMBEDDING_DIM, NUM_CLASSES, LETTER_LABELS,
    VisionBridgeLetterBaseModel,
)

assert INPUT_DIM == 126, INPUT_DIM
assert HIDDEN_DIM == 128, HIDDEN_DIM
assert EMBEDDING_DIM == 64, EMBEDDING_DIM
assert NUM_CLASSES == 26, NUM_CLASSES
assert tuple(LETTER_LABELS) == tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

model = VisionBridgeLetterBaseModel()
assert model.input_dim == 126
assert model.hidden_dim == 128
assert model.embedding_dim == 64
assert model.num_classes == 26

layers = list(model.encoder.children())
layer_names = [type(layer).__name__ for layer in layers]
expected = ["LayerNorm", "Linear", "GELU", "Dropout", "Linear", "LayerNorm", "GELU"]
assert layer_names == expected, layer_names

sample = torch.zeros(2, 126)
with torch.inference_mode():
    embedding = model.embed(sample)
    logits = model(sample)
assert embedding.shape == (2, 64), embedding.shape
assert logits.shape == (2, 26), logits.shape
assert torch.isfinite(embedding).all()
assert torch.isfinite(logits).all()

print("V3 architecture contract: PASS")
print("input_dim =", model.input_dim)
print("hidden_dim =", model.hidden_dim)
print("embedding_dim =", model.embedding_dim)
print("num_classes =", model.num_classes)
print("encoder =", " -> ".join(layer_names))
print("logits_shape =", tuple(logits.shape))
'''

result = subprocess.run(
    [str(PYTHON), "-c", contract],
    check=False, env=ENV, text=True, capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"V3 architecture contract failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )

In [ ]:
# Cell 8 — Train letter base model (up to 1000 epochs; early-stop on target)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"
ENV["PYTHONUNBUFFERED"] = "1"

if not PYTHON.exists():
    raise RuntimeError("Training Python is missing. Run Cell 1.")
if not REPO.exists():
    raise RuntimeError("VisionBridge repository is missing. Run Cell 1.")

DATA_DIR = Path("/content/visionbridge_letter_data")
required = [DATA_DIR / "train.npz", DATA_DIR / "val.npz", DATA_DIR / "test.npz", DATA_DIR / "labels.json"]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise RuntimeError("Prepared dataset is incomplete. Run Cells 5 and 6.\n" + "\n".join(missing))

weights_dir = REPO / "backend/app/models/weights"
weights_dir.mkdir(parents=True, exist_ok=True)

command = [
    str(PYTHON), "-m", "app.training.letter_base",
    "--data-dir", str(DATA_DIR),
    "--output", str(weights_dir / "letter_base_model.pt"),
    "--epochs", "1000",
    "--batch-size", "128",
    "--lr", "0.001",
    "--weight-decay", "0.0001",
    "--target-class-accuracy", "1.0",
    "--seed", "42",
    "--hidden-dim", "128",
    "--embedding-dim", "64",
    "--dropout", "0.10",
]

print("Starting training...", flush=True)
result = subprocess.run(command, check=False, env=ENV, text=True, capture_output=True)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Training failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
print("Training command finished successfully.")

In [ ]:
# Cell 9 — Verify checkpoint loads
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"

if not PYTHON.exists():
    raise RuntimeError("Training Python is missing. Run Cell 1.")
if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("Training did not create a checkpoint. Run Cell 8.")

ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

check = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
import sys
from app.models.letter_model import load_checkpoint

model = load_checkpoint(sys.argv[1])
print("CHECKPOINT LOAD: PASS")
print("input_dim =", model.input_dim)
print("hidden_dim =", model.hidden_dim)
print("embedding_dim =", model.embedding_dim)
print("classes =", model.num_classes)
print("labels =", "".join(model.labels))
'''

result = subprocess.run(
    [str(PYTHON), "-c", check, str(CHECKPOINT)],
    check=False, env=ENV, text=True, capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Checkpoint load failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )

print("Checkpoint bytes:", CHECKPOINT.stat().st_size)
print("Training pipeline: PASS")
print()
print("Download this file from Colab:")
print(" ", CHECKPOINT)

In [ ]:
# Cell 10 — Evaluate V3 on the untouched test split
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"
DATA_DIR = Path("/content/visionbridge_letter_data")
OUTPUT_JSON = Path("/content/visionbridge_letter_evaluation.json")

if not PYTHON.exists():
    raise RuntimeError("Training Python is missing. Run Cell 1.")
if not REPO.is_dir():
    raise RuntimeError("VisionBridge repository is missing. Run Cell 1.")
if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("V3 checkpoint is missing/empty. Run Cell 8.")
if not DATA_DIR.is_dir():
    raise RuntimeError("Prepared landmark dataset is missing. Run Cells 5 and 6.")

ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

command = [
    str(PYTHON), "-m", "app.training.evaluate_letter_base",
    "--checkpoint", str(CHECKPOINT),
    "--data-dir", str(DATA_DIR),
    "--split", "test",
    "--output-json", str(OUTPUT_JSON),
]

print("Evaluating V3 on the untouched RealSign test split...")
result = subprocess.run(command, check=False, env=ENV, text=True, capture_output=True)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        "V3 evaluation failed.\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
if not OUTPUT_JSON.is_file():
    raise RuntimeError("V3 evaluation finished but the JSON report was not created.")

print("\nV3 test evaluation: PASS")
print("Evaluation report:", OUTPUT_JSON)

In [ ]:
# Cell 11 — Build the V3 release evidence manifest
from pathlib import Path
import hashlib
import json

REPO = Path("/content/VisionBridge")
CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"
EVIDENCE = Path("/content/visionbridge_v3_release_evidence.json")
EVALUATION = Path("/content/visionbridge_letter_evaluation.json")

if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("V3 checkpoint is missing")
if not EVALUATION.is_file():
    raise RuntimeError("V3 evaluation report is missing")

digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
report = json.loads(EVALUATION.read_text(encoding="utf-8"))
per_letter = report["per_letter_accuracy"]
if set(per_letter) != set("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    raise RuntimeError("Evaluation report does not contain all A-Z classes")
if any(int(count) <= 0 for count in report["class_counts"].values()):
    raise RuntimeError("Evaluation report contains an empty test class")
if report["model"]["model_version"] != "visionbridge-letter-base-v3":
    raise RuntimeError("Evaluation report is not for the active V3 model")

evidence = {
    "model_version": "visionbridge-letter-base-v3",
    "preprocessing_version": report["model"]["preprocessing_version"],
    "landmark_runtime": report["model"]["landmark_runtime"],
    "training": {
        "epochs": 1000,
        "batch_size": 128,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "target_class_accuracy": 1.0,
        "seed": 42,
        "hidden_dim": 128,
        "embedding_dim": 64,
        "dropout": 0.10,
    },
    "dataset_source": {
        "source_repository": "https://github.com/RealSign62/RealSign-Indian-Sign-Language-Dataset",
        "source_commit": "17c51dcc158b7b6359b7c6667edc51c7f148f3f4",
        "source_url": (
            "https://media.githubusercontent.com/media/"
            "RealSign62/RealSign-Indian-Sign-Language-Dataset/"
            "17c51dcc158b7b6359b7c6667edc51c7f148f3f4/Dataset.zip"
        ),
        "size": 656_689_688,
        "sha256": "008cae248e346b8c31fbbea057fcc3f69c6909d29a88e6bb1fb0369f528de2b5",
    },
    "checkpoint": {
        "path": str(CHECKPOINT),
        "bytes": CHECKPOINT.stat().st_size,
        "sha256": digest,
    },
    "evaluation": report,
    "signer_independent_status": (
        "BLOCKED: verified signer IDs are not exposed by the active source folder structure"
    ),
}
EVIDENCE.write_text(json.dumps(evidence, indent=2), encoding="utf-8")

print("V3 release evidence: PASS")
print("checkpoint_sha256 =", digest)
print("test_accuracy =", report["overall_accuracy"])
print("macro_accuracy =", report["macro_accuracy"])
print("worst_letter =", report["worst_letter"], report["worst_letter_accuracy"])
print("evidence =", EVIDENCE)
print()
print("=== DONE ===")
print("Download checkpoint:")
print(" ", CHECKPOINT)
print("Download evaluation:")
print(" ", EVALUATION)
print("Download evidence:")
print(" ", EVIDENCE)

## Result

After a successful run, download:

| File | Path |
|------|------|
| Checkpoint | `/content/VisionBridge/backend/app/models/weights/letter_base_model.pt` |
| Test evaluation | `/content/visionbridge_letter_evaluation.json` |
| Release evidence | `/content/visionbridge_v3_release_evidence.json` |

Copy the checkpoint into your local repo at:

```
backend/app/models/weights/letter_base_model.pt
```

If anything fails: **Runtime → Restart runtime**, then run from Cell 1 again and read the printed stderr.